In [ ]:
#!/usr/bin/env python
"""Experimento completo de embeddings no HateBR em um único arquivo.

Comandos:
    python projeto_pln.py listar-modelos
    python projeto_pln.py auditar
    python projeto_pln.py executar

Exemplo completo:
    python projeto_pln.py executar \
        --modelos e5_base e5_large_instruct minilm_multilingual bertimbau_sts \
        --condicoes sem_instrucao manual_en manual_pt

Para instruções automáticas, defina GROQ_API_KEY no ambiente e acrescente:
    --condicoes sem_instrucao manual_en manual_pt automatica_en automatica_pt
"""

from __future__ import annotations

import argparse
import gc
import hashlib
import json
import os
import platform
import re
import sys
import time
from collections.abc import Iterable, Sequence
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

In [ ]:
# =============================================================================
# 1. CONFIGURAÇÃO
# =============================================================================

SEED = 42
TEXT_COL = "comentario"
LABEL_COL = "label_final"
ID_COL = "id"

SEM_INSTRUCAO = "sem_instrucao"
MANUAL_EN = "manual_en"
MANUAL_PT = "manual_pt"
AUTOMATICA_EN = "automatica_en"
AUTOMATICA_PT = "automatica_pt"

TODAS_CONDICOES = (
    SEM_INSTRUCAO,
    MANUAL_EN,
    MANUAL_PT,
    AUTOMATICA_EN,
    AUTOMATICA_PT,
)
CONDICOES_INSTRUCIONAIS = frozenset(
    {MANUAL_EN, MANUAL_PT, AUTOMATICA_EN, AUTOMATICA_PT}
)

REQUIRED_COLUMNS = {
    ID_COL,
    TEXT_COL,
    "anotator1",
    "anotator2",
    "anotator3",
    LABEL_COL,
    "links_post",
    "account_post",
}

MANUAL_INSTRUCTIONS = {
    "en": (
        "Represent the Brazilian Portuguese comment for hate speech detection. "
        "Consider insults, discrimination, prejudice, attacks on protected groups, "
        "irony, implicit hostility, and offensive language."
    ),
    "pt": (
        "Represente o comentário em português brasileiro para detecção de discurso "
        "de ódio. Considere insultos, discriminação, preconceito, ataques a grupos "
        "protegidos, ironia, hostilidade implícita e linguagem ofensiva."
    ),
}


@dataclass(frozen=True, slots=True)
class ModelConfig:
    """Protocolo e metadados de um encoder."""

    name: str
    hf_id: str
    input_style: str
    supports_instructions: bool
    language_scope: str
    dimensions: int
    max_tokens: int
    role: str
    source_url: str

    def supports_condition(self, condition: str) -> bool:
        if condition == SEM_INSTRUCAO:
            return True
        return self.supports_instructions and condition in CONDICOES_INSTRUCIONAIS


MODEL_REGISTRY: dict[str, ModelConfig] = {
    "e5_base": ModelConfig(
        name="e5_base",
        hf_id="intfloat/multilingual-e5-base",
        input_style="e5_query",
        supports_instructions=False,
        language_scope="Multilíngue (100 idiomas; inclui português via XLM-R)",
        dimensions=768,
        max_tokens=512,
        role="Controle E5 multilíngue sem instrução livre",
        source_url="https://huggingface.co/intfloat/multilingual-e5-base",
    ),
    "e5_large_instruct": ModelConfig(
        name="e5_large_instruct",
        hf_id="intfloat/multilingual-e5-large-instruct",
        input_style="e5_instruct",
        supports_instructions=True,
        language_scope="Multilíngue (100 idiomas; inclui português via XLM-R)",
        dimensions=1024,
        max_tokens=512,
        role="Modelo instrucional principal",
        source_url="https://huggingface.co/intfloat/multilingual-e5-large-instruct",
    ),
    "minilm_multilingual": ModelConfig(
        name="minilm_multilingual",
        hf_id="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
        input_style="plain",
        supports_instructions=False,
        language_scope="Multilíngue (50 idiomas)",
        dimensions=384,
        max_tokens=128,
        role="Controle multilíngue leve para execução rápida",
        source_url=(
            "https://huggingface.co/sentence-transformers/"
            "paraphrase-multilingual-MiniLM-L12-v2"
        ),
    ),
    "mpnet_multilingual": ModelConfig(
        name="mpnet_multilingual",
        hf_id="sentence-transformers/paraphrase-multilingual-mpnet-base-v2",
        input_style="plain",
        supports_instructions=False,
        language_scope="Multilíngue (50 idiomas)",
        dimensions=768,
        max_tokens=128,
        role="Controle multilíngue independente da família E5",
        source_url=(
            "https://huggingface.co/sentence-transformers/"
            "paraphrase-multilingual-mpnet-base-v2"
        ),
    ),
    "bertimbau_sts": ModelConfig(
        name="bertimbau_sts",
        hf_id="rufimelo/bert-large-portuguese-cased-sts",
        input_style="plain",
        supports_instructions=False,
        language_scope="Português brasileiro",
        dimensions=1024,
        max_tokens=128,
        role="Controle monolíngue ajustado para similaridade textual",
        source_url="https://huggingface.co/rufimelo/bert-large-portuguese-cased-sts",
    ),
    "bge_m3": ModelConfig(
        name="bge_m3",
        hf_id="BAAI/bge-m3",
        input_style="plain",
        supports_instructions=False,
        language_scope="Multilíngue (mais de 100 idiomas)",
        dimensions=1024,
        max_tokens=8192,
        role="Controle multilíngue moderno e independente",
        source_url="https://huggingface.co/BAAI/bge-m3",
    ),
}

DEFAULT_MODELS = ("e5_base", "e5_large_instruct", "minilm_multilingual")
DEFAULT_CONDITIONS = (SEM_INSTRUCAO, MANUAL_EN, MANUAL_PT)

In [ ]:
# =============================================================================
# 2. DADOS E DIVISÃO REPRODUZÍVEL
# =============================================================================


@dataclass(slots=True)
class Dataset:
    frame: pd.DataFrame
    sha256: str
    path: Path
    audit: dict[str, Any]


def file_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def load_dataset(path: str | Path) -> Dataset:
    """Carrega o HateBR e interrompe em caso de dados incompatíveis."""

    path = Path(path).resolve()
    if not path.exists():
        raise FileNotFoundError(f"Base não encontrada: {path}")

    frame = pd.read_csv(path)
    missing_columns = REQUIRED_COLUMNS - set(frame.columns)
    if missing_columns:
        raise ValueError(
            "Colunas obrigatórias ausentes: " + ", ".join(sorted(missing_columns))
        )

    null_required = frame[[ID_COL, TEXT_COL, LABEL_COL]].isna().any(axis=1)
    blank_text = frame[TEXT_COL].fillna("").astype(str).str.strip().eq("")
    invalid = null_required | blank_text
    cleaned = frame.loc[~invalid].copy().reset_index(drop=True)

    if cleaned.empty:
        raise ValueError("A base ficou vazia depois da validação.")
    if cleaned[ID_COL].duplicated().any():
        raise ValueError("A coluna 'id' precisa ser única.")

    labels = set(cleaned[LABEL_COL].unique().tolist())
    if labels != {0, 1}:
        raise ValueError(f"Esperava rótulos binários 0/1, mas encontrei: {labels}")

    annotator_cols = ["anotator1", "anotator2", "anotator3"]
    unanimous = cleaned[annotator_cols].nunique(axis=1).eq(1)
    majority = cleaned[annotator_cols].sum(axis=1).ge(2).astype(int)

    audit: dict[str, Any] = {
        "rows_original": len(frame),
        "rows_valid": len(cleaned),
        "rows_removed": int(invalid.sum()),
        "class_counts": {
            str(key): int(value)
            for key, value in cleaned[LABEL_COL].value_counts().sort_index().items()
        },
        "duplicate_comments": int(cleaned[TEXT_COL].duplicated(keep=False).sum()),
        "unanimous_annotations": int(unanimous.sum()),
        "non_unanimous_annotations": int((~unanimous).sum()),
        "final_label_equals_majority": int((majority == cleaned[LABEL_COL]).sum()),
        "unique_posts": int(cleaned["links_post"].nunique()),
        "unique_accounts": int(cleaned["account_post"].nunique()),
    }
    return Dataset(
        frame=cleaned,
        sha256=file_sha256(path),
        path=path,
        audit=audit,
    )


def create_or_load_splits(
    dataset: Dataset,
    output_dir: str | Path,
    seed: int = SEED,
) -> dict[str, pd.DataFrame]:
    """Cria 70/15/15 estratificado e persiste os mesmos IDs para todos os modelos."""

    try:
        from sklearn.model_selection import train_test_split
    except ImportError as exc:
        raise RuntimeError(
            "scikit-learn não instalado. Execute: pip install -r requirements.txt"
        ) from exc

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    manifest_path = output_dir / "split_manifest.csv"
    metadata_path = output_dir / "split_manifest.json"

    if manifest_path.exists() and metadata_path.exists():
        metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
        if metadata.get("dataset_sha256") == dataset.sha256 and metadata.get("seed") == seed:
            manifest = pd.read_csv(manifest_path)
            expected_ids = set(dataset.frame[ID_COL].tolist())
            if set(manifest[ID_COL].tolist()) == expected_ids:
                indexed = dataset.frame.set_index(ID_COL, drop=False)
                return {
                    split: indexed.loc[
                        manifest.loc[manifest["split"] == split, ID_COL].tolist()
                    ].reset_index(drop=True)
                    for split in ("train", "valid", "test")
                }

    train, temporary = train_test_split(
        dataset.frame,
        test_size=0.30,
        stratify=dataset.frame[LABEL_COL],
        random_state=seed,
    )
    valid, test = train_test_split(
        temporary,
        test_size=0.50,
        stratify=temporary[LABEL_COL],
        random_state=seed,
    )
    splits = {
        "train": train.reset_index(drop=True),
        "valid": valid.reset_index(drop=True),
        "test": test.reset_index(drop=True),
    }

    manifest = pd.concat(
        [part[[ID_COL]].assign(split=split) for split, part in splits.items()],
        ignore_index=True,
    )
    manifest.to_csv(manifest_path, index=False)
    metadata_path.write_text(
        json.dumps(
            {
                "dataset_sha256": dataset.sha256,
                "seed": seed,
                "sizes": {key: len(value) for key, value in splits.items()},
            },
            ensure_ascii=False,
            indent=2,
        ),
        encoding="utf-8",
    )
    return splits


In [ ]:
# =============================================================================
# 3. INSTRUÇÕES AUTOMÁTICAS
# =============================================================================


def atomic_json_write(path: Path, value: object) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(value, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    temporary.replace(path)


def extract_json_array(raw: str) -> list[str]:
    cleaned = raw.strip()
    if cleaned.startswith("```"):
        cleaned = cleaned.strip("`").strip()
        if cleaned.lower().startswith("json"):
            cleaned = cleaned[4:].strip()
    parsed = json.loads(cleaned)
    if not isinstance(parsed, list) or not all(isinstance(item, str) for item in parsed):
        raise ValueError("A LLM não retornou um array JSON de strings.")
    result = [item.strip() for item in parsed]
    if any(not item for item in result):
        raise ValueError("A LLM retornou uma instrução vazia.")
    return result


def instruction_prompt(texts: list[str], language: str) -> str:
    numbered = "\n".join(f"{index + 1}. {text}" for index, text in enumerate(texts))
    if language == "pt":
        return (
            "Você receberá comentários em português brasileiro. Para cada comentário, "
            "gere uma instrução curta (máximo de 25 palavras), em português, que diga a "
            "um modelo de embeddings quais indícios observar para detectar discurso de "
            "ódio. Não decida o rótulo. Retorne apenas um array JSON de strings, na "
            "mesma ordem e com o mesmo tamanho da entrada.\n\n" + numbered
        )
    if language == "en":
        return (
            "You will receive Brazilian Portuguese comments. For each comment, generate "
            "one short instruction (at most 25 words), in English, telling an embedding "
            "model which cues to inspect for hate-speech detection. Do not decide the "
            "label. Return only a JSON string array in the same order and with the same "
            "length as the input.\n\n" + numbered
        )
    raise ValueError(f"Idioma de instrução inválido: {language}")


def generate_automatic_instructions(
    frames: Iterable[pd.DataFrame],
    language: str,
    cache_path: str | Path,
    model: str = "llama-3.1-8b-instant",
    batch_size: int = 10,
    max_retries: int = 4,
    rpm_limit: int = 25,
) -> dict[int, str]:
    """Gera instruções por lote sem substituir falhas por baseline."""

    api_key = os.environ.get("GROQ_API_KEY")
    if not api_key:
        raise RuntimeError(
            "GROQ_API_KEY não definida. A chave deve ficar no ambiente, nunca no código."
        )
    try:
        import requests
    except ImportError as exc:
        raise RuntimeError(
            "requests não instalado. Execute: pip install -r requirements.txt"
        ) from exc

    cache_path = Path(cache_path)
    if cache_path.exists():
        raw_cache = json.loads(cache_path.read_text(encoding="utf-8"))
        cache = {int(key): str(value).strip() for key, value in raw_cache.items()}
    else:
        cache = {}

    records = pd.concat(list(frames), ignore_index=True)[[ID_COL, TEXT_COL]]
    pending = [
        (int(row[ID_COL]), str(row[TEXT_COL]))
        for _, row in records.iterrows()
        if not cache.get(int(row[ID_COL]), "").strip()
    ]
    batches = [pending[i : i + batch_size] for i in range(0, len(pending), batch_size)]
    minimum_interval = 60.0 / max(1, rpm_limit)
    last_call = 0.0

    print(
        f"[instruções {language}] {len(pending)} pendentes em {len(batches)} lotes"
    )
    for batch_number, batch in enumerate(batches, start=1):
        ids = [item[0] for item in batch]
        texts = [item[1] for item in batch]
        payload = {
            "model": model,
            "messages": [
                {"role": "user", "content": instruction_prompt(texts, language)}
            ],
            "temperature": 0.0,
            "max_tokens": 60 * len(texts),
        }
        error: Exception | None = None
        for attempt in range(max_retries):
            wait = minimum_interval - (time.monotonic() - last_call)
            if wait > 0:
                time.sleep(wait)
            last_call = time.monotonic()
            try:
                response = requests.post(
                    "https://api.groq.com/openai/v1/chat/completions",
                    headers={"Authorization": f"Bearer {api_key}"},
                    json=payload,
                    timeout=90,
                )
                if response.status_code == 429:
                    time.sleep(5 * (attempt + 1))
                    continue
                response.raise_for_status()
                result = extract_json_array(
                    response.json()["choices"][0]["message"]["content"]
                )
                if len(result) != len(batch):
                    raise ValueError(
                        f"Esperava {len(batch)} instruções; recebi {len(result)}."
                    )
                cache.update(dict(zip(ids, result)))
                atomic_json_write(cache_path, {str(k): v for k, v in cache.items()})
                error = None
                break
            except (
                requests.RequestException,
                json.JSONDecodeError,
                KeyError,
                TypeError,
                ValueError,
            ) as exc:
                error = exc
                time.sleep(2 * (attempt + 1))
        if error is not None:
            raise RuntimeError(
                f"Falha no lote {batch_number}/{len(batches)}; o cache parcial foi "
                f"preservado. Motivo: {error}"
            ) from error

    missing = [
        int(identifier)
        for identifier in records[ID_COL]
        if not cache.get(int(identifier), "").strip()
    ]
    if missing:
        raise RuntimeError(
            f"Cache incompleto: faltam {len(missing)} instruções. "
            "O experimento não continuará para evitar mistura com o baseline."
        )
    return cache



In [ ]:
# =============================================================================
# 4. FORMATAÇÃO E CACHE DE EMBEDDINGS
# =============================================================================


def input_digest(values: Sequence[str]) -> str:
    digest = hashlib.sha256()
    for value in values:
        encoded = value.encode("utf-8")
        digest.update(len(encoded).to_bytes(8, "big"))
        digest.update(encoded)
    return digest.hexdigest()


def slug(value: str) -> str:
    return re.sub(r"[^a-zA-Z0-9_.-]+", "_", value).strip("_")


def format_inputs(
    config: ModelConfig,
    condition: str,
    texts: Sequence[str],
    automatic_instructions: Sequence[str] | None = None,
) -> list[str]:
    """Aplica exatamente o formato esperado por cada família de modelo."""

    if not config.supports_condition(condition):
        raise ValueError(
            f"{config.name} não foi treinado para a condição '{condition}'. "
            "Modelos não instrucionais só podem ser usados em 'sem_instrucao'."
        )

    if condition == SEM_INSTRUCAO:
        if config.input_style in {"e5_query", "e5_instruct"}:
            return [f"query: {text}" for text in texts]
        return list(texts)

    if config.input_style != "e5_instruct":
        raise ValueError(f"Estilo instrucional não implementado: {config.input_style}")

    if condition == MANUAL_EN:
        instructions = [MANUAL_INSTRUCTIONS["en"]] * len(texts)
    elif condition == MANUAL_PT:
        instructions = [MANUAL_INSTRUCTIONS["pt"]] * len(texts)
    elif condition in {AUTOMATICA_EN, AUTOMATICA_PT}:
        if automatic_instructions is None:
            raise ValueError(f"Instruções ausentes para a condição '{condition}'.")
        if len(automatic_instructions) != len(texts):
            raise ValueError("Quantidade de instruções diferente da quantidade de textos.")
        instructions = [str(item).strip() for item in automatic_instructions]
        if any(not item for item in instructions):
            raise ValueError(
                "Há instruções automáticas vazias. O baseline não será usado como fallback."
            )
    else:
        raise ValueError(f"Condição desconhecida: {condition}")

    return [
        f"Instruct: {instruction}\nQuery: {text}"
        for instruction, text in zip(instructions, texts)
    ]


def load_sentence_transformer(config: ModelConfig, device: str | None = None):
    try:
        from sentence_transformers import SentenceTransformer
    except ImportError as exc:
        raise RuntimeError(
            "sentence-transformers não instalado. Use Python 3.10-3.12 e execute: "
            "pip install -r requirements.txt"
        ) from exc

    kwargs: dict[str, Any] = {"trust_remote_code": False}
    if device and device != "auto":
        kwargs["device"] = device
    return SentenceTransformer(config.hf_id, **kwargs)


def resolved_model_revision(model: object) -> str:
    try:
        transformer = model[0]
        revision = transformer.auto_model.config._commit_hash
        return str(revision or "unknown")
    except (AttributeError, IndexError, TypeError):
        return "unknown"


class EmbeddingCache:
    """Cache que recusa embeddings de outro modelo/configuração/texto."""

    def __init__(self, root: str | Path):
        self.root = Path(root)

    def paths(
        self,
        dataset_sha256: str,
        config: ModelConfig,
        condition: str,
        split: str,
    ) -> tuple[Path, Path]:
        directory = (
            self.root
            / "v2"
            / dataset_sha256[:16]
            / slug(config.hf_id)
            / condition
        )
        return directory / f"{split}.npy", directory / f"{split}.json"

    def load(
        self,
        dataset_sha256: str,
        config: ModelConfig,
        condition: str,
        split: str,
        formatted_inputs: Sequence[str],
    ) -> np.ndarray | None:
        array_path, metadata_path = self.paths(
            dataset_sha256, config, condition, split
        )
        if not array_path.exists() or not metadata_path.exists():
            return None
        try:
            metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
            expected = {
                "model_id": config.hf_id,
                "condition": condition,
                "split": split,
                "dataset_sha256": dataset_sha256,
                "input_sha256": input_digest(formatted_inputs),
                "rows": len(formatted_inputs),
            }
            if any(metadata.get(key) != value for key, value in expected.items()):
                return None
            array = np.load(array_path, allow_pickle=False)
            if (
                array.ndim != 2
                or array.shape[0] != len(formatted_inputs)
                or not np.isfinite(array).all()
            ):
                return None
            return array
        except (OSError, ValueError, json.JSONDecodeError):
            return None

    def save(
        self,
        array: np.ndarray,
        dataset_sha256: str,
        config: ModelConfig,
        condition: str,
        split: str,
        formatted_inputs: Sequence[str],
        model_revision: str,
    ) -> None:
        array_path, metadata_path = self.paths(
            dataset_sha256, config, condition, split
        )
        array_path.parent.mkdir(parents=True, exist_ok=True)
        array_tmp = array_path.with_suffix(".npy.tmp")
        metadata_tmp = metadata_path.with_suffix(".json.tmp")
        with array_tmp.open("wb") as stream:
            np.save(stream, np.asarray(array, dtype=np.float32), allow_pickle=False)
        metadata = {
            "model_id": config.hf_id,
            "model_revision": model_revision,
            "condition": condition,
            "split": split,
            "dataset_sha256": dataset_sha256,
            "input_sha256": input_digest(formatted_inputs),
            "rows": len(formatted_inputs),
            "dimensions": int(array.shape[1]),
            "normalized": True,
        }
        metadata_tmp.write_text(
            json.dumps(metadata, ensure_ascii=False, indent=2),
            encoding="utf-8",
        )
        os.replace(array_tmp, array_path)
        os.replace(metadata_tmp, metadata_path)


def get_or_encode(
    model: object,
    config: ModelConfig,
    cache: EmbeddingCache,
    dataset_sha256: str,
    condition: str,
    split: str,
    texts: Sequence[str],
    automatic_instructions: Sequence[str] | None,
    batch_size: int,
    force: bool = False,
) -> np.ndarray:
    formatted = format_inputs(
        config,
        condition,
        texts,
        automatic_instructions=automatic_instructions,
    )
    if not force:
        cached = cache.load(dataset_sha256, config, condition, split, formatted)
        if cached is not None:
            print(f"[cache] {config.name} | {condition} | {split}")
            return cached

    embeddings = model.encode(
        formatted,
        batch_size=batch_size,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True,
    )
    embeddings = np.asarray(embeddings, dtype=np.float32)
    if embeddings.ndim != 2 or embeddings.shape[0] != len(texts):
        raise RuntimeError(
            f"Formato inesperado de embeddings: {embeddings.shape}; "
            f"esperava {len(texts)} linhas."
        )
    if not np.isfinite(embeddings).all():
        raise RuntimeError("O modelo produziu valores NaN ou infinitos.")

    cache.save(
        embeddings,
        dataset_sha256,
        config,
        condition,
        split,
        formatted,
        model_revision=resolved_model_revision(model),
    )
    return embeddings



In [ ]:
# =============================================================================
# 5. CLASSIFICAÇÃO E MÉTRICAS
# =============================================================================


@dataclass(slots=True)
class EvaluationResult:
    metrics: dict[str, float]
    predictions: np.ndarray
    best_c: float


def classification_metrics(
    y_true: np.ndarray,
    y_pred: np.ndarray,
) -> dict[str, float]:
    try:
        from sklearn.metrics import (
            accuracy_score,
            balanced_accuracy_score,
            f1_score,
            matthews_corrcoef,
            precision_score,
            recall_score,
        )
    except ImportError as exc:
        raise RuntimeError(
            "scikit-learn não instalado. Execute: pip install -r requirements.txt"
        ) from exc

    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "precision_weighted": float(
            precision_score(y_true, y_pred, average="weighted", zero_division=0)
        ),
        "recall_weighted": float(
            recall_score(y_true, y_pred, average="weighted", zero_division=0)
        ),
        "f1_weighted": float(
            f1_score(y_true, y_pred, average="weighted", zero_division=0)
        ),
        "f1_macro": float(
            f1_score(y_true, y_pred, average="macro", zero_division=0)
        ),
        "mcc": float(matthews_corrcoef(y_true, y_pred)),
    }


def evaluate_linear_probe(
    x_train: np.ndarray,
    y_train: np.ndarray,
    x_valid: np.ndarray,
    y_valid: np.ndarray,
    x_test: np.ndarray,
    y_test: np.ndarray,
    seed: int = SEED,
    c_values: tuple[float, ...] = (0.1, 1.0, 10.0),
    bootstrap_samples: int = 500,
) -> EvaluationResult:
    """Seleciona C na validação e avalia o teste uma única vez."""

    try:
        from sklearn.linear_model import LogisticRegression
        from sklearn.metrics import f1_score
    except ImportError as exc:
        raise RuntimeError(
            "scikit-learn não instalado. Execute: pip install -r requirements.txt"
        ) from exc

    best_c = c_values[0]
    best_score = -1.0
    for c_value in c_values:
        classifier = LogisticRegression(
            C=c_value,
            max_iter=2000,
            random_state=seed,
            solver="lbfgs",
        )
        classifier.fit(x_train, y_train)
        score = f1_score(
            y_valid,
            classifier.predict(x_valid),
            average="macro",
            zero_division=0,
        )
        if score > best_score:
            best_score = float(score)
            best_c = c_value

    classifier = LogisticRegression(
        C=best_c,
        max_iter=2000,
        random_state=seed,
        solver="lbfgs",
    )
    classifier.fit(
        np.concatenate([x_train, x_valid], axis=0),
        np.concatenate([y_train, y_valid], axis=0),
    )
    predictions = classifier.predict(x_test)
    metrics = classification_metrics(y_test, predictions)

    if bootstrap_samples > 0:
        from sklearn.metrics import f1_score

        rng = np.random.default_rng(seed)
        scores = np.empty(bootstrap_samples, dtype=np.float64)
        for index in range(bootstrap_samples):
            sample = rng.integers(0, len(y_test), size=len(y_test))
            scores[index] = f1_score(
                y_test[sample],
                predictions[sample],
                average="macro",
                zero_division=0,
            )
        metrics["f1_macro_ci95_low"] = float(np.quantile(scores, 0.025))
        metrics["f1_macro_ci95_high"] = float(np.quantile(scores, 0.975))

    return EvaluationResult(
        metrics=metrics,
        predictions=np.asarray(predictions),
        best_c=float(best_c),
    )



In [ ]:
# =============================================================================
# 6. PIPELINE COMPLETO
# =============================================================================


def release_accelerator_cache() -> None:
    gc.collect()
    try:
        import torch

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except ImportError:
        pass


def validate_model_names(names: list[str] | tuple[str, ...]) -> list[str]:
    unknown = sorted(set(names) - MODEL_REGISTRY.keys())
    if unknown:
        raise ValueError(
            f"Modelo(s) desconhecido(s): {', '.join(unknown)}. "
            f"Opções: {', '.join(MODEL_REGISTRY)}"
        )
    return list(dict.fromkeys(names))


def validate_conditions(conditions: list[str] | tuple[str, ...]) -> list[str]:
    unknown = sorted(set(conditions) - set(TODAS_CONDICOES))
    if unknown:
        raise ValueError(
            f"Condição(ões) desconhecida(s): {', '.join(unknown)}. "
            f"Opções: {', '.join(TODAS_CONDICOES)}"
        )
    return list(dict.fromkeys(conditions))


def run_experiment(
    data_path: str | Path,
    cache_dir: str | Path,
    output_dir: str | Path,
    model_names: list[str],
    conditions: list[str],
    seed: int = SEED,
    batch_size: int = 32,
    device: str = "auto",
    force_embeddings: bool = False,
    groq_model: str = "llama-3.1-8b-instant",
    bootstrap_samples: int = 500,
) -> pd.DataFrame:
    """Executa embeddings, classificador, métricas e arquivos de saída."""

    started = time.time()
    output_dir = Path(output_dir)
    cache_dir = Path(cache_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    cache_dir.mkdir(parents=True, exist_ok=True)

    dataset = load_dataset(data_path)
    splits = create_or_load_splits(dataset, output_dir=output_dir, seed=seed)
    print(
        "Divisão:",
        " | ".join(f"{name}={len(frame)}" for name, frame in splits.items()),
    )

    instruction_maps: dict[str, dict[int, str]] = {}
    for language, condition in (("en", AUTOMATICA_EN), ("pt", AUTOMATICA_PT)):
        condition_is_applicable = condition in conditions and any(
            MODEL_REGISTRY[name].supports_condition(condition)
            for name in model_names
        )
        if condition_is_applicable:
            instruction_maps[language] = generate_automatic_instructions(
                splits.values(),
                language=language,
                cache_path=(
                    cache_dir
                    / "v2"
                    / dataset.sha256[:16]
                    / f"automatic_instructions_{language}.json"
                ),
                model=groq_model,
            )

    cache = EmbeddingCache(cache_dir)
    result_rows: list[dict[str, object]] = []
    prediction_frames: list[pd.DataFrame] = []

    for model_name in model_names:
        config = MODEL_REGISTRY[model_name]
        applicable = [
            condition for condition in conditions if config.supports_condition(condition)
        ]
        skipped = [condition for condition in conditions if condition not in applicable]
        for condition in skipped:
            print(f"[ignorado] {model_name} | {condition}: modelo não instrucional")
        if not applicable:
            continue

        print(f"\nCarregando {model_name}: {config.hf_id}")
        model = load_sentence_transformer(config, device=device)
        try:
            for condition in applicable:
                arrays: dict[str, np.ndarray] = {}
                for split_name, frame in splits.items():
                    language = (
                        "en"
                        if condition == AUTOMATICA_EN
                        else "pt"
                        if condition == AUTOMATICA_PT
                        else None
                    )
                    automatic = None
                    if language:
                        instruction_map = instruction_maps[language]
                        automatic = [
                            instruction_map[int(identifier)]
                            for identifier in frame[ID_COL]
                        ]
                    arrays[split_name] = get_or_encode(
                        model=model,
                        config=config,
                        cache=cache,
                        dataset_sha256=dataset.sha256,
                        condition=condition,
                        split=split_name,
                        texts=frame[TEXT_COL].astype(str).tolist(),
                        automatic_instructions=automatic,
                        batch_size=batch_size,
                        force=force_embeddings,
                    )

                evaluation = evaluate_linear_probe(
                    arrays["train"],
                    splits["train"][LABEL_COL].to_numpy(),
                    arrays["valid"],
                    splits["valid"][LABEL_COL].to_numpy(),
                    arrays["test"],
                    splits["test"][LABEL_COL].to_numpy(),
                    seed=seed,
                    bootstrap_samples=bootstrap_samples,
                )
                row: dict[str, object] = {
                    "modelo": model_name,
                    "model_id": config.hf_id,
                    "condicao": condition,
                    "dimensoes": int(arrays["train"].shape[1]),
                    "melhor_C": evaluation.best_c,
                    **evaluation.metrics,
                }
                result_rows.append(row)
                prediction_frames.append(
                    pd.DataFrame(
                        {
                            ID_COL: splits["test"][ID_COL].to_numpy(),
                            LABEL_COL: splits["test"][LABEL_COL].to_numpy(),
                            "predicao": evaluation.predictions,
                            "modelo": model_name,
                            "condicao": condition,
                        }
                    )
                )
                print(
                    f"[resultado] {model_name} | {condition}: "
                    f"F1-macro={evaluation.metrics['f1_macro']:.4f}"
                )
        finally:
            del model
            release_accelerator_cache()

    if not result_rows:
        raise RuntimeError("Nenhuma combinação modelo/condição pôde ser executada.")

    results = pd.DataFrame(result_rows).sort_values(
        ["f1_macro", "modelo", "condicao"],
        ascending=[False, True, True],
    )
    results.to_csv(output_dir / "resultados_comparativos.csv", index=False)
    pd.concat(prediction_frames, ignore_index=True).to_csv(
        output_dir / "predicoes_teste.csv",
        index=False,
    )
    (output_dir / "execucao.json").write_text(
        json.dumps(
            {
                "dataset": str(dataset.path),
                "dataset_sha256": dataset.sha256,
                "dataset_audit": dataset.audit,
                "models": model_names,
                "conditions": conditions,
                "seed": seed,
                "batch_size": batch_size,
                "device": device,
                "python": sys.version,
                "platform": platform.platform(),
                "elapsed_seconds": round(time.time() - started, 3),
            },
            ensure_ascii=False,
            indent=2,
        ),
        encoding="utf-8",
    )
    return results


In [ ]:
# =============================================================================
# 7. INTERFACE DE LINHA DE COMANDO
# =============================================================================


def project_root() -> Path:
    return Path.cwd()


def build_parser() -> argparse.ArgumentParser:
    root = project_root()
    parser = argparse.ArgumentParser(
        description="Comparação reproduzível de embeddings no HateBR."
    )
    subparsers = parser.add_subparsers(dest="command", required=True)
    subparsers.add_parser(
        "listar-modelos",
        help="Mostra modelos, suporte linguístico e uso de instruções.",
    )

    audit = subparsers.add_parser("auditar", help="Valida e resume a base HateBR.")
    audit.add_argument("--dados", type=Path, default=root / "HateBR.csv")

    run = subparsers.add_parser("executar", help="Gera embeddings e compara resultados.")
    run.add_argument("--dados", type=Path, default=root / "HateBR.csv")
    run.add_argument("--cache", type=Path, default=root / "cache")
    run.add_argument("--saida", type=Path, default=root / "outputs")
    run.add_argument(
        "--modelos",
        nargs=+",
        default=list(DEFAULT_MODELS),
        help="Nomes do registro. Use 'listar-modelos' para ver opções.",
    )
    run.add_argument(
        "--condicoes",
        nargs=+",
        default=list(DEFAULT_CONDITIONS),
        choices=TODAS_CONDICOES,
    )
    run.add_argument("--seed", type=int, default=SEED)
    run.add_argument("--batch-size", type=int, default=32)
    run.add_argument("--dispositivo", default="auto", choices=("auto", "cpu", "cuda"))
    run.add_argument("--forcar-embeddings", action="store_true")
    run.add_argument("--groq-model", default="llama-3.1-8b-instant")
    run.add_argument("--bootstrap", type=int, default=500)
    return parser


def main(argv: list[str] | None = None) -> int:
    parser = build_parser()
    args = parser.parse_args(argv)

    if args.command == "listar-modelos":
        for config in MODEL_REGISTRY.values():
            instruction = "sim" if config.supports_instructions else "não"
            print(
                f"{config.name}\n"
                f"  ID: {config.hf_id}\n"
                f"  Idiomas: {config.language_scope}\n"
                f"  Instruções livres: {instruction}\n"
                f"  Papel: {config.role}\n"
                f"  Fonte: {config.source_url}\n"
            )
        return 0

    if args.command == "auditar":
        dataset = load_dataset(args.dados)
        print(json.dumps(dataset.audit, ensure_ascii=False, indent=2))
        print(f"sha256: {dataset.sha256}")
        return 0

    if args.command == "executar":
        model_names = validate_model_names(args.modelos)
        conditions = validate_conditions(args.condicoes)
        results = run_experiment(
            data_path=args.dados,
            cache_dir=args.cache,
            output_dir=args.saida,
            model_names=model_names,
            conditions=conditions,
            seed=args.seed,
            batch_size=args.batch_size,
            device=args.dispositivo,
            force_embeddings=args.forcar_embeddings,
            groq_model=args.groq_model,
            bootstrap_samples=args.bootstrap,
        )
        display_columns = [
            "modelo",
            "condicao",
            "f1_macro",
            "f1_macro_ci95_low",
            "f1_macro_ci95_high",
            "accuracy",
        ]
        print("\nRanking final:")
        print(
            results[
                [column for column in display_columns if column in results.columns]
            ].to_string(index=False)
        )
        return 0

    parser.error("Comando inválido.")
    return 2


if __name__ == "__main__":
    # In Colab, sys.argv often contains unexpected values. Pass an empty list to main.
    # For command-line execution, you would call main(sys.argv[1:]).
    # To run the experiment, you need to provide a command like 'executar' to the main function.
    # Example: raise SystemExit(main(argv=['executar', '--dados', 'HateBR.csv']))
    raise SystemExit(main(argv=['executar']))

SyntaxError: unterminated string literal (detected at line 30) (3278592988.py, line 30)

In [ ]:
#!/usr/bin/env python
"""Experimento completo de embeddings no HateBR em um único arquivo.

Comandos:
    python projeto_pln.py listar-modelos
    python projeto_pln.py auditar
    python projeto_pln.py executar

Exemplo completo:
    python projeto_pln.py executar \
        --modelos e5_base e5_large_instruct minilm_multilingual bertimbau_sts \
        --condicoes sem_instrucao manual_en manual_pt

Para instruções automáticas, defina GROQ_API_KEY no ambiente e acrescente:
    --condicoes sem_instrucao manual_en manual_pt automatica_en automatica_pt
"""

from __future__ import annotations

import argparse
import gc
import hashlib
import importlib.util
import json
import os
import platform
import re
import subprocess
import sys
import time
from collections.abc import Iterable, Sequence
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd

# =============================================================================
# 1. CONFIGURAÇÃO
# =============================================================================

SEED = 42
TEXT_COL = "comentario"
LABEL_COL = "label_final"
ID_COL = "id"

SEM_INSTRUCAO = "sem_instrucao"
MANUAL_EN = "manual_en"
MANUAL_PT = "manual_pt"
AUTOMATICA_EN = "automatica_en"
AUTOMATICA_PT = "automatica_pt"

TODAS_CONDICOES = (
    SEM_INSTRUCAO,
    MANUAL_EN,
    MANUAL_PT,
    AUTOMATICA_EN,
    AUTOMATICA_PT,
)
CONDICOES_INSTRUCIONAIS = frozenset(
    {MANUAL_EN, MANUAL_PT, AUTOMATICA_EN, AUTOMATICA_PT}
)

REQUIRED_COLUMNS = {
    ID_COL,
    TEXT_COL,
    "anotator1",
    "anotator2",
    "anotator3",
    LABEL_COL,
    "links_post",
    "account_post",
}

MANUAL_INSTRUCTIONS = {
    "en": (
        "Represent the Brazilian Portuguese comment for hate speech detection. "
        "Consider insults, discrimination, prejudice, attacks on protected groups, "
        "irony, implicit hostility, and offensive language."
    ),
    "pt": (
        "Represente o comentário em português brasileiro para detecção de discurso "
        "de ódio. Considere insultos, discriminação, preconceito, ataques a grupos "
        "protegidos, ironia, hostilidade implícita e linguagem ofensiva."
    ),
}


@dataclass(frozen=True, slots=True)
class ModelConfig:
    """Protocolo e metadados de um encoder."""

    name: str
    hf_id: str
    input_style: str
    supports_instructions: bool
    language_scope: str
    dimensions: int
    max_tokens: int
    role: str
    source_url: str

    def supports_condition(self, condition: str) -> bool:
        if condition == SEM_INSTRUCAO:
            return True
        return self.supports_instructions and condition in CONDICOES_INSTRUCIONAIS


MODEL_REGISTRY: dict[str, ModelConfig] = {
    "e5_base": ModelConfig(
        name="e5_base",
        hf_id="intfloat/multilingual-e5-base",
        input_style="e5_query",
        supports_instructions=False,
        language_scope="Multilíngue (100 idiomas; inclui português via XLM-R)",
        dimensions=768,
        max_tokens=512,
        role="Controle E5 multilíngue sem instrução livre",
        source_url="https://huggingface.co/intfloat/multilingual-e5-base",
    ),
    "e5_large_instruct": ModelConfig(
        name="e5_large_instruct",
        hf_id="intfloat/multilingual-e5-large-instruct",
        input_style="e5_instruct",
        supports_instructions=True,
        language_scope="Multilíngue (100 idiomas; inclui português via XLM-R)",
        dimensions=1024,
        max_tokens=512,
        role="Modelo instrucional principal",
        source_url="https://huggingface.co/intfloat/multilingual-e5-large-instruct",
    ),
    "minilm_multilingual": ModelConfig(
        name="minilm_multilingual",
        hf_id="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
        input_style="plain",
        supports_instructions=False,
        language_scope="Multilíngue (50 idiomas)",
        dimensions=384,
        max_tokens=128,
        role="Controle multilíngue leve para execução rápida",
        source_url=(
            "https://huggingface.co/sentence-transformers/"
            "paraphrase-multilingual-MiniLM-L12-v2"
        ),
    ),
    "mpnet_multilingual": ModelConfig(
        name="mpnet_multilingual",
        hf_id="sentence-transformers/paraphrase-multilingual-mpnet-base-v2",
        input_style="plain",
        supports_instructions=False,
        language_scope="Multilíngue (50 idiomas)",
        dimensions=768,
        max_tokens=128,
        role="Controle multilíngue independente da família E5",
        source_url=(
            "https://huggingface.co/sentence-transformers/"
            "paraphrase-multilingual-mpnet-base-v2"
        ),
    ),
    "bertimbau_sts": ModelConfig(
        name="bertimbau_sts",
        hf_id="rufimelo/bert-large-portuguese-cased-sts",
        input_style="plain",
        supports_instructions=False,
        language_scope="Português brasileiro",
        dimensions=1024,
        max_tokens=128,
        role="Controle monolíngue ajustado para similaridade textual",
        source_url="https://huggingface.co/rufimelo/bert-large-portuguese-cased-sts",
    ),
    "bge_m3": ModelConfig(
        name="bge_m3",
        hf_id="BAAI/bge-m3",
        input_style="plain",
        supports_instructions=False,
        language_scope="Multilíngue (mais de 100 idiomas)",
        dimensions=1024,
        max_tokens=8192,
        role="Controle multilíngue moderno e independente",
        source_url="https://huggingface.co/BAAI/bge-m3",
    ),
}

DEFAULT_MODELS = ("e5_base", "e5_large_instruct", "minilm_multilingual")
DEFAULT_CONDITIONS = (SEM_INSTRUCAO, MANUAL_EN, MANUAL_PT)


# =============================================================================
# 2. DADOS E DIVISÃO REPRODUZÍVEL
# =============================================================================


@dataclass(slots=True)
class Dataset:
    frame: pd.DataFrame
    sha256: str
    path: Path
    audit: dict[str, Any]


def file_sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def load_dataset(path: str | Path) -> Dataset:
    """Carrega o HateBR e interrompe em caso de dados incompatíveis."""

    path = Path(path).resolve()
    if not path.exists():
        raise FileNotFoundError(f"Base não encontrada: {path}")

    frame = pd.read_csv(path)
    missing_columns = REQUIRED_COLUMNS - set(frame.columns)
    if missing_columns:
        raise ValueError(
            "Colunas obrigatórias ausentes: " + ", ".join(sorted(missing_columns))
        )

    null_required = frame[[ID_COL, TEXT_COL, LABEL_COL]].isna().any(axis=1)
    blank_text = frame[TEXT_COL].fillna("").astype(str).str.strip().eq("")
    invalid = null_required | blank_text
    cleaned = frame.loc[~invalid].copy().reset_index(drop=True)

    if cleaned.empty:
        raise ValueError("A base ficou vazia depois da validação.")
    if cleaned[ID_COL].duplicated().any():
        raise ValueError("A coluna 'id' precisa ser única.")

    labels = set(cleaned[LABEL_COL].unique().tolist())
    if labels != {0, 1}:
        raise ValueError(f"Esperava rótulos binários 0/1, mas encontrei: {labels}")

    annotator_cols = ["anotator1", "anotator2", "anotator3"]
    unanimous = cleaned[annotator_cols].nunique(axis=1).eq(1)
    majority = cleaned[annotator_cols].sum(axis=1).ge(2).astype(int)

    audit: dict[str, Any] = {
        "rows_original": len(frame),
        "rows_valid": len(cleaned),
        "rows_removed": int(invalid.sum()),
        "class_counts": {
            str(key): int(value)
            for key, value in cleaned[LABEL_COL].value_counts().sort_index().items()
        },
        "duplicate_comments": int(cleaned[TEXT_COL].duplicated(keep=False).sum()),
        "unanimous_annotations": int(unanimous.sum()),
        "non_unanimous_annotations": int((~unanimous).sum()),
        "final_label_equals_majority": int((majority == cleaned[LABEL_COL]).sum()),
        "unique_posts": int(cleaned["links_post"].nunique()),
        "unique_accounts": int(cleaned["account_post"].nunique()),
    }
    return Dataset(
        frame=cleaned,
        sha256=file_sha256(path),
        path=path,
        audit=audit,
    )


def create_or_load_splits(
    dataset: Dataset,
    output_dir: str | Path,
    seed: int = SEED,
) -> dict[str, pd.DataFrame]:
    """Cria 70/15/15 estratificado e persiste os mesmos IDs para todos os modelos."""

    try:
        from sklearn.model_selection import train_test_split
    except ImportError as exc:
        raise RuntimeError(
            "scikit-learn não instalado. Execute: pip install -r requirements.txt"
        ) from exc

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    manifest_path = output_dir / "split_manifest.csv"
    metadata_path = output_dir / "split_manifest.json"

    if manifest_path.exists() and metadata_path.exists():
        metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
        if metadata.get("dataset_sha256") == dataset.sha256 and metadata.get("seed") == seed:
            manifest = pd.read_csv(manifest_path)
            expected_ids = set(dataset.frame[ID_COL].tolist())
            if set(manifest[ID_COL].tolist()) == expected_ids:
                indexed = dataset.frame.set_index(ID_COL, drop=False)
                return {
                    split: indexed.loc[
                        manifest.loc[manifest["split"] == split, ID_COL].tolist()
                    ].reset_index(drop=True)
                    for split in ("train", "valid", "test")
                }

    train, temporary = train_test_split(
        dataset.frame,
        test_size=0.30,
        stratify=dataset.frame[LABEL_COL],
        random_state=seed,
    )
    valid, test = train_test_split(
        temporary,
        test_size=0.50,
        stratify=temporary[LABEL_COL],
        random_state=seed,
    )
    splits = {
        "train": train.reset_index(drop=True),
        "valid": valid.reset_index(drop=True),
        "test": test.reset_index(drop=True),
    }

    manifest = pd.concat(
        [part[[ID_COL]].assign(split=split) for split, part in splits.items()],
        ignore_index=True,
    )
    manifest.to_csv(manifest_path, index=False)
    metadata_path.write_text(
        json.dumps(
            {
                "dataset_sha256": dataset.sha256,
                "seed": seed,
                "sizes": {key: len(value) for key, value in splits.items()},
            },
            ensure_ascii=False,
            indent=2,
        ),
        encoding="utf-8",
    )
    return splits


# =============================================================================
# 3. INSTRUÇÕES AUTOMÁTICAS
# =============================================================================


def atomic_json_write(path: Path, value: object) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(
        json.dumps(value, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    temporary.replace(path)


def extract_json_array(raw: str) -> list[str]:
    cleaned = raw.strip()
    if cleaned.startswith("```"):
        cleaned = cleaned.strip("`").strip()
        if cleaned.lower().startswith("json"):
            cleaned = cleaned[4:].strip()
    parsed = json.loads(cleaned)
    if not isinstance(parsed, list) or not all(isinstance(item, str) for item in parsed):
        raise ValueError("A LLM não retornou um array JSON de strings.")
    result = [item.strip() for item in parsed]
    if any(not item for item in result):
        raise ValueError("A LLM retornou uma instrução vazia.")
    return result


def instruction_prompt(texts: list[str], language: str) -> str:
    numbered = "\n".join(f"{index + 1}. {text}" for index, text in enumerate(texts))
    if language == "pt":
        return (
            "Você receberá comentários em português brasileiro. Para cada comentário, "
            "gere uma instrução curta (máximo de 25 palavras), em português, que diga a "
            "um modelo de embeddings quais indícios observar para detectar discurso de "
            "ódio. Não decida o rótulo. Retorne apenas um array JSON de strings, na "
            "mesma ordem e com o mesmo tamanho da entrada.\n\n" + numbered
        )
    if language == "en":
        return (
            "You will receive Brazilian Portuguese comments. For each comment, generate "
            "one short instruction (at most 25 words), in English, telling an embedding "
            "model which cues to inspect for hate-speech detection. Do not decide the "
            "label. Return only a JSON string array in the same order and with the same "
            "length as the input.\n\n" + numbered
        )
    raise ValueError(f"Idioma de instrução inválido: {language}")


def generate_automatic_instructions(
    frames: Iterable[pd.DataFrame],
    language: str,
    cache_path: str | Path,
    model: str = "llama-3.1-8b-instant",
    batch_size: int = 10,
    max_retries: int = 4,
    rpm_limit: int = 25,
) -> dict[int, str]:
    """Gera instruções por lote sem substituir falhas por baseline."""

    api_key = os.environ.get("GROQ_API_KEY")
    if not api_key:
        raise RuntimeError(
            "GROQ_API_KEY não definida. A chave deve ficar no ambiente, nunca no código."
        )
    try:
        import requests
    except ImportError as exc:
        raise RuntimeError(
            "requests não instalado. Execute: pip install -r requirements.txt"
        ) from exc

    cache_path = Path(cache_path)
    if cache_path.exists():
        raw_cache = json.loads(cache_path.read_text(encoding="utf-8"))
        cache = {int(key): str(value).strip() for key, value in raw_cache.items()}
    else:
        cache = {}

    records = pd.concat(list(frames), ignore_index=True)[[ID_COL, TEXT_COL]]
    pending = [
        (int(row[ID_COL]), str(row[TEXT_COL]))
        for _, row in records.iterrows()
        if not cache.get(int(row[ID_COL]), "").strip()
    ]
    batches = [pending[i : i + batch_size] for i in range(0, len(pending), batch_size)]
    minimum_interval = 60.0 / max(1, rpm_limit)
    last_call = 0.0

    print(
        f"[instruções {language}] {len(pending)} pendentes em {len(batches)} lotes"
    )
    for batch_number, batch in enumerate(batches, start=1):
        ids = [item[0] for item in batch]
        texts = [item[1] for item in batch]
        payload = {
            "model": model,
            "messages": [
                {"role": "user", "content": instruction_prompt(texts, language)}
            ],
            "temperature": 0.0,
            "max_tokens": 60 * len(texts),
        }
        error: Exception | None = None
        for attempt in range(max_retries):
            wait = minimum_interval - (time.monotonic() - last_call)
            if wait > 0:
                time.sleep(wait)
            last_call = time.monotonic()
            try:
                response = requests.post(
                    "https://api.groq.com/openai/v1/chat/completions",
                    headers={"Authorization": f"Bearer {api_key}"},
                    json=payload,
                    timeout=90,
                )
                if response.status_code == 429:
                    time.sleep(5 * (attempt + 1))
                    continue
                response.raise_for_status()
                result = extract_json_array(
                    response.json()["choices"][0]["message"]["content"]
                )
                if len(result) != len(batch):
                    raise ValueError(
                        f"Esperava {len(batch)} instruções; recebi {len(result)}."
                    )
                cache.update(dict(zip(ids, result)))
                atomic_json_write(cache_path, {str(k): v for k, v in cache.items()})
                error = None
                break
            except (
                requests.RequestException,
                json.JSONDecodeError,
                KeyError,
                TypeError,
                ValueError,
            ) as exc:
                error = exc
                time.sleep(2 * (attempt + 1))
        if error is not None:
            raise RuntimeError(
                f"Falha no lote {batch_number}/{len(batches)}; o cache parcial foi "
                f"preservado. Motivo: {error}"
            ) from error

    missing = [
        int(identifier)
        for identifier in records[ID_COL]
        if not cache.get(int(identifier), "").strip()
    ]
    if missing:
        raise RuntimeError(
            f"Cache incompleto: faltam {len(missing)} instruções. "
            "O experimento não continuará para evitar mistura com o baseline."
        )
    return cache


# =============================================================================
# 4. FORMATAÇÃO E CACHE DE EMBEDDINGS
# =============================================================================


def input_digest(values: Sequence[str]) -> str:
    digest = hashlib.sha256()
    for value in values:
        encoded = value.encode("utf-8")
        digest.update(len(encoded).to_bytes(8, "big"))
        digest.update(encoded)
    return digest.hexdigest()


def slug(value: str) -> str:
    return re.sub(r"[^a-zA-Z0-9_.-]+", "_", value).strip("_")


def format_inputs(
    config: ModelConfig,
    condition: str,
    texts: Sequence[str],
    automatic_instructions: Sequence[str] | None = None,
) -> list[str]:
    """Aplica exatamente o formato esperado por cada família de modelo."""

    if not config.supports_condition(condition):
        raise ValueError(
            f"{config.name} não foi treinado para a condição '{condition}'. "
            "Modelos não instrucionais só podem ser usados em 'sem_instrucao'."
        )

    if condition == SEM_INSTRUCAO:
        if config.input_style in {"e5_query", "e5_instruct"}:
            return [f"query: {text}" for text in texts]
        return list(texts)

    if config.input_style != "e5_instruct":
        raise ValueError(f"Estilo instrucional não implementado: {config.input_style}")

    if condition == MANUAL_EN:
        instructions = [MANUAL_INSTRUCTIONS["en"]] * len(texts)
    elif condition == MANUAL_PT:
        instructions = [MANUAL_INSTRUCTIONS["pt"]] * len(texts)
    elif condition in {AUTOMATICA_EN, AUTOMATICA_PT}:
        if automatic_instructions is None:
            raise ValueError(f"Instruções ausentes para a condição '{condition}'.")
        if len(automatic_instructions) != len(texts):
            raise ValueError("Quantidade de instruções diferente da quantidade de textos.")
        instructions = [str(item).strip() for item in automatic_instructions]
        if any(not item for item in instructions):
            raise ValueError(
                "Há instruções automáticas vazias. O baseline não será usado como fallback."
            )
    else:
        raise ValueError(f"Condição desconhecida: {condition}")

    return [
        f"Instruct: {instruction}\nQuery: {text}"
        for instruction, text in zip(instructions, texts)
    ]


def load_sentence_transformer(config: ModelConfig, device: str | None = None):
    try:
        from sentence_transformers import SentenceTransformer
    except ImportError as exc:
        raise RuntimeError(
            "sentence-transformers não instalado. Use Python 3.10-3.12 e execute: "
            "pip install -r requirements.txt"
        ) from exc

    kwargs: dict[str, Any] = {"trust_remote_code": False}
    if device and device != "auto":
        kwargs["device"] = device
    return SentenceTransformer(config.hf_id, **kwargs)


def resolved_model_revision(model: object) -> str:
    try:
        transformer = model[0]
        revision = transformer.auto_model.config._commit_hash
        return str(revision or "unknown")
    except (AttributeError, IndexError, TypeError):
        return "unknown"


class EmbeddingCache:
    """Cache que recusa embeddings de outro modelo/configuração/texto."""

    def __init__(self, root: str | Path):
        self.root = Path(root)

    def paths(
        self,
        dataset_sha256: str,
        config: ModelConfig,
        condition: str,
        split: str,
    ) -> tuple[Path, Path]:
        directory = (
            self.root
            / "v2"
            / dataset_sha256[:16]
            / slug(config.hf_id)
            / condition
        )
        return directory / f"{split}.npy", directory / f"{split}.json"

    def load(
        self,
        dataset_sha256: str,
        config: ModelConfig,
        condition: str,
        split: str,
        formatted_inputs: Sequence[str],
    ) -> np.ndarray | None:
        array_path, metadata_path = self.paths(
            dataset_sha256, config, condition, split
        )
        if not array_path.exists() or not metadata_path.exists():
            return None
        try:
            metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
            expected = {
                "model_id": config.hf_id,
                "condition": condition,
                "split": split,
                "dataset_sha256": dataset_sha256,
                "input_sha256": input_digest(formatted_inputs),
                "rows": len(formatted_inputs),
            }
            if any(metadata.get(key) != value for key, value in expected.items()):
                return None
            array = np.load(array_path, allow_pickle=False)
            if (
                array.ndim != 2
                or array.shape[0] != len(formatted_inputs)
                or not np.isfinite(array).all()
            ):
                return None
            return array
        except (OSError, ValueError, json.JSONDecodeError):
            return None

    def save(
        self,
        array: np.ndarray,
        dataset_sha256: str,
        config: ModelConfig,
        condition: str,
        split: str,
        formatted_inputs: Sequence[str],
        model_revision: str,
    ) -> None:
        array_path, metadata_path = self.paths(
            dataset_sha256, config, condition, split
        )
        array_path.parent.mkdir(parents=True, exist_ok=True)
        array_tmp = array_path.with_suffix(".npy.tmp")
        metadata_tmp = metadata_path.with_suffix(".json.tmp")
        with array_tmp.open("wb") as stream:
            np.save(stream, np.asarray(array, dtype=np.float32), allow_pickle=False)
        metadata = {
            "model_id": config.hf_id,
            "model_revision": model_revision,
            "condition": condition,
            "split": split,
            "dataset_sha256": dataset_sha256,
            "input_sha256": input_digest(formatted_inputs),
            "rows": len(formatted_inputs),
            "dimensions": int(array.shape[1]),
            "normalized": True,
        }
        metadata_tmp.write_text(
            json.dumps(metadata, ensure_ascii=False, indent=2),
            encoding="utf-8",
        )
        os.replace(array_tmp, array_path)
        os.replace(metadata_tmp, metadata_path)


def get_or_encode(
    model: object,
    config: ModelConfig,
    cache: EmbeddingCache,
    dataset_sha256: str,
    condition: str,
    split: str,
    texts: Sequence[str],
    automatic_instructions: Sequence[str] | None,
    batch_size: int,
    force: bool = False,
) -> np.ndarray:
    formatted = format_inputs(
        config,
        condition,
        texts,
        automatic_instructions=automatic_instructions,
    )
    if not force:
        cached = cache.load(dataset_sha256, config, condition, split, formatted)
        if cached is not None:
            print(f"[cache] {config.name} | {condition} | {split}")
            return cached

    embeddings = model.encode(
        formatted,
        batch_size=batch_size,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True,
    )
    embeddings = np.asarray(embeddings, dtype=np.float32)
    if embeddings.ndim != 2 or embeddings.shape[0] != len(texts):
        raise RuntimeError(
            f"Formato inesperado de embeddings: {embeddings.shape}; "
            f"esperava {len(texts)} linhas."
        )
    if not np.isfinite(embeddings).all():
        raise RuntimeError("O modelo produziu valores NaN ou infinitos.")

    cache.save(
        embeddings,
        dataset_sha256,
        config,
        condition,
        split,
        formatted,
        model_revision=resolved_model_revision(model),
    )
    return embeddings


# =============================================================================
# 5. CLASSIFICAÇÃO E MÉTRICAS
# =============================================================================


@dataclass(slots=True)
class EvaluationResult:
    metrics: dict[str, float]
    predictions: np.ndarray
    best_c: float


def classification_metrics(
    y_true: np.ndarray,
    y_pred: np.ndarray,
) -> dict[str, float]:
    try:
        from sklearn.metrics import (
            accuracy_score,
            balanced_accuracy_score,
            f1_score,
            matthews_corrcoef,
            precision_score,
            recall_score,
        )
    except ImportError as exc:
        raise RuntimeError(
            "scikit-learn não instalado. Execute: pip install -r requirements.txt"
        ) from exc

    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "precision_weighted": float(
            precision_score(y_true, y_pred, average="weighted", zero_division=0)
        ),
        "recall_weighted": float(
            recall_score(y_true, y_pred, average="weighted", zero_division=0)
        ),
        "f1_weighted": float(
            f1_score(y_true, y_pred, average="weighted", zero_division=0)
        ),
        "f1_macro": float(
            f1_score(y_true, y_pred, average="macro", zero_division=0)
        ),
        "mcc": float(matthews_corrcoef(y_true, y_pred)),
    }


def evaluate_linear_probe(
    x_train: np.ndarray,
    y_train: np.ndarray,
    x_valid: np.ndarray,
    y_valid: np.ndarray,
    x_test: np.ndarray,
    y_test: np.ndarray,
    seed: int = SEED,
    c_values: tuple[float, ...] = (0.1, 1.0, 10.0),
    bootstrap_samples: int = 500,
) -> EvaluationResult:
    """Seleciona C na validação e avalia o teste uma única vez."""

    try:
        from sklearn.linear_model import LogisticRegression
        from sklearn.metrics import f1_score
    except ImportError as exc:
        raise RuntimeError(
            "scikit-learn não instalado. Execute: pip install -r requirements.txt"
        ) from exc

    best_c = c_values[0]
    best_score = -1.0
    for c_value in c_values:
        classifier = LogisticRegression(
            C=c_value,
            max_iter=2000,
            random_state=seed,
            solver="lbfgs",
        )
        classifier.fit(x_train, y_train)
        score = f1_score(
            y_valid,
            classifier.predict(x_valid),
            average="macro",
            zero_division=0,
        )
        if score > best_score:
            best_score = float(score)
            best_c = c_value

    classifier = LogisticRegression(
        C=best_c,
        max_iter=2000,
        random_state=seed,
        solver="lbfgs",
    )
    classifier.fit(
        np.concatenate([x_train, x_valid], axis=0),
        np.concatenate([y_train, y_valid], axis=0),
    )
    predictions = classifier.predict(x_test)
    metrics = classification_metrics(y_test, predictions)

    if bootstrap_samples > 0:
        from sklearn.metrics import f1_score

        rng = np.random.default_rng(seed)
        scores = np.empty(bootstrap_samples, dtype=np.float64)
        for index in range(bootstrap_samples):
            sample = rng.integers(0, len(y_test), size=len(y_test))
            scores[index] = f1_score(
                y_test[sample],
                predictions[sample],
                average="macro",
                zero_division=0,
            )
        metrics["f1_macro_ci95_low"] = float(np.quantile(scores, 0.025))
        metrics["f1_macro_ci95_high"] = float(np.quantile(scores, 0.975))

    return EvaluationResult(
        metrics=metrics,
        predictions=np.asarray(predictions),
        best_c=float(best_c),
    )


# =============================================================================
# 6. PIPELINE COMPLETO
# =============================================================================


def release_accelerator_cache() -> None:
    gc.collect()
    try:
        import torch

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except ImportError:
        pass


def validate_model_names(names: list[str] | tuple[str, ...]) -> list[str]:
    unknown = sorted(set(names) - MODEL_REGISTRY.keys())
    if unknown:
        raise ValueError(
            f"Modelo(s) desconhecido(s): {', '.join(unknown)}. "
            f"Opções: {', '.join(MODEL_REGISTRY)}"
        )
    return list(dict.fromkeys(names))


def validate_conditions(conditions: list[str] | tuple[str, ...]) -> list[str]:
    unknown = sorted(set(conditions) - set(TODAS_CONDICOES))
    if unknown:
        raise ValueError(
            f"Condição(ões) desconhecida(s): {', '.join(unknown)}. "
            f"Opções: {', '.join(TODAS_CONDICOES)}"
        )
    return list(dict.fromkeys(conditions))


def run_experiment(
    data_path: str | Path,
    cache_dir: str | Path,
    output_dir: str | Path,
    model_names: list[str],
    conditions: list[str],
    seed: int = SEED,
    batch_size: int = 32,
    device: str = "auto",
    force_embeddings: bool = False,
    groq_model: str = "llama-3.1-8b-instant",
    bootstrap_samples: int = 500,
) -> pd.DataFrame:
    """Executa embeddings, classificador, métricas e arquivos de saída."""

    started = time.time()
    output_dir = Path(output_dir)
    cache_dir = Path(cache_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    cache_dir.mkdir(parents=True, exist_ok=True)

    dataset = load_dataset(data_path)
    splits = create_or_load_splits(dataset, output_dir=output_dir, seed=seed)
    print(
        "Divisão:",
        " | ".join(f"{name}={len(frame)}" for name, frame in splits.items()),
    )

    instruction_maps: dict[str, dict[int, str]] = {}
    for language, condition in (("en", AUTOMATICA_EN), ("pt", AUTOMATICA_PT)):
        condition_is_applicable = condition in conditions and any(
            MODEL_REGISTRY[name].supports_condition(condition)
            for name in model_names
        )
        if condition_is_applicable:
            instruction_maps[language] = generate_automatic_instructions(
                splits.values(),
                language=language,
                cache_path=(
                    cache_dir
                    / "v2"
                    / dataset.sha256[:16]
                    / f"automatic_instructions_{language}.json"
                ),
                model=groq_model,
            )

    cache = EmbeddingCache(cache_dir)
    result_rows: list[dict[str, object]] = []
    prediction_frames: list[pd.DataFrame] = []

    for model_name in model_names:
        config = MODEL_REGISTRY[model_name]
        applicable = [
            condition for condition in conditions if config.supports_condition(condition)
        ]
        skipped = [condition for condition in conditions if condition not in applicable]
        for condition in skipped:
            print(f"[ignorado] {model_name} | {condition}: modelo não instrucional")
        if not applicable:
            continue

        print(f"\nCarregando {model_name}: {config.hf_id}")
        model = load_sentence_transformer(config, device=device)
        try:
            for condition in applicable:
                arrays: dict[str, np.ndarray] = {}
                for split_name, frame in splits.items():
                    language = (
                        "en"
                        if condition == AUTOMATICA_EN
                        else "pt"
                        if condition == AUTOMATICA_PT
                        else None
                    )
                    automatic = None
                    if language:
                        instruction_map = instruction_maps[language]
                        automatic = [
                            instruction_map[int(identifier)]
                            for identifier in frame[ID_COL]
                        ]
                    arrays[split_name] = get_or_encode(
                        model=model,
                        config=config,
                        cache=cache,
                        dataset_sha256=dataset.sha256,
                        condition=condition,
                        split=split_name,
                        texts=frame[TEXT_COL].astype(str).tolist(),
                        automatic_instructions=automatic,
                        batch_size=batch_size,
                        force=force_embeddings,
                    )

                evaluation = evaluate_linear_probe(
                    arrays["train"],
                    splits["train"][LABEL_COL].to_numpy(),
                    arrays["valid"],
                    splits["valid"][LABEL_COL].to_numpy(),
                    arrays["test"],
                    splits["test"][LABEL_COL].to_numpy(),
                    seed=seed,
                    bootstrap_samples=bootstrap_samples,
                )
                row: dict[str, object] = {
                    "modelo": model_name,
                    "model_id": config.hf_id,
                    "condicao": condition,
                    "dimensoes": int(arrays["train"].shape[1]),
                    "melhor_C": evaluation.best_c,
                    **evaluation.metrics,
                }
                result_rows.append(row)
                prediction_frames.append(
                    pd.DataFrame(
                        {
                            ID_COL: splits["test"][ID_COL].to_numpy(),
                            LABEL_COL: splits["test"][LABEL_COL].to_numpy(),
                            "predicao": evaluation.predictions,
                            "modelo": model_name,
                            "condicao": condition,
                        }
                    )
                )
                print(
                    f"[resultado] {model_name} | {condition}: "
                    f"F1-macro={evaluation.metrics['f1_macro']:.4f}"
                )
        finally:
            del model
            release_accelerator_cache()

    if not result_rows:
        raise RuntimeError("Nenhuma combinação modelo/condição pôde ser executada.")

    results = pd.DataFrame(result_rows).sort_values(
        ["f1_macro", "modelo", "condicao"],
        ascending=[False, True, True],
    )
    results.to_csv(output_dir / "resultados_comparativos.csv", index=False)
    pd.concat(prediction_frames, ignore_index=True).to_csv(
        output_dir / "predicoes_teste.csv",
        index=False,
    )
    (output_dir / "execucao.json").write_text(
        json.dumps(
            {
                "dataset": str(dataset.path),
                "dataset_sha256": dataset.sha256,
                "dataset_audit": dataset.audit,
                "models": model_names,
                "conditions": conditions,
                "seed": seed,
                "batch_size": batch_size,
                "device": device,
                "python": sys.version,
                "platform": platform.platform(),
                "elapsed_seconds": round(time.time() - started, 3),
            },
            ensure_ascii=False,
            indent=2,
        ),
        encoding="utf-8",
    )
    return results


# =============================================================================
# 7. SUPORTE DIRETO AO GOOGLE COLAB
# =============================================================================


def project_root() -> Path:
    """Pasta do script ou diretório atual quando executado como célula."""

    script_path = globals().get("__file__")
    if script_path:
        return Path(script_path).resolve().parent
    return Path.cwd().resolve()


def install_missing_dependencies() -> None:
    """Instala somente pacotes ausentes; útil quando o código é colado no Colab."""

    dependency_map = {
        "sklearn": "scikit-learn>=1.5,<2",
        "sentence_transformers": "sentence-transformers>=3,<6",
        "requests": "requests>=2.32,<3",
        "tqdm": "tqdm>=4.66,<5",
    }
    missing = [
        package
        for module, package in dependency_map.items()
        if importlib.util.find_spec(module) is None
    ]
    if not missing:
        print("Dependências: OK")
        return

    print("Instalando dependências ausentes:", ", ".join(missing))
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "--quiet", *missing]
    )
    print("Dependências instaladas.")


def executar_colab(
    dados: str | Path = "HateBR.csv",
    modelos: Sequence[str] = DEFAULT_MODELS,
    condicoes: Sequence[str] = DEFAULT_CONDITIONS,
    cache: str | Path = "cache",
    saida: str | Path = "outputs",
    dispositivo: str = "auto",
    batch_size: int = 32,
    seed: int = SEED,
    bootstrap: int = 500,
    instalar_dependencias: bool = True,
    forcar_embeddings: bool = False,
    groq_model: str = "llama-3.1-8b-instant",
) -> pd.DataFrame:
    """Executa o experimento diretamente de uma célula do Google Colab.

    Exemplo:
        resultados = executar_colab(
            modelos=["e5_base", "e5_large_instruct", "minilm_multilingual"],
            condicoes=["sem_instrucao", "manual_en", "manual_pt"],
        )
    """

    if instalar_dependencias:
        install_missing_dependencies()

    root = project_root()

    def resolve_from_root(value: str | Path) -> Path:
        path = Path(value)
        return path if path.is_absolute() else root / path

    data_path = resolve_from_root(dados)
    if not data_path.exists():
        available_csv = sorted(root.glob("*.csv"))
        csv_message = (
            " CSVs encontrados: " + ", ".join(path.name for path in available_csv)
            if available_csv
            else " Nenhum CSV foi encontrado nessa pasta."
        )
        raise FileNotFoundError(
            f"Não encontrei {data_path}. Faça upload de HateBR.csv para {root}."
            + csv_message
        )

    return run_experiment(
        data_path=data_path,
        cache_dir=resolve_from_root(cache),
        output_dir=resolve_from_root(saida),
        model_names=validate_model_names(list(modelos)),
        conditions=validate_conditions(list(condicoes)),
        seed=seed,
        batch_size=batch_size,
        device=dispositivo,
        force_embeddings=forcar_embeddings,
        groq_model=groq_model,
        bootstrap_samples=bootstrap,
    )


def print_colab_instructions() -> None:
    print(
        "\nCódigo carregado no Google Colab.\n"
        "Agora execute em uma nova célula:\n\n"
        "resultados = executar_colab()\n"
        "display(resultados)\n\n"
        "O arquivo HateBR.csv precisa estar na pasta /content."
    )


# =============================================================================
# 8. INTERFACE DE LINHA DE COMANDO
# =============================================================================


def build_parser() -> argparse.ArgumentParser:
    root = project_root()
    parser = argparse.ArgumentParser(
        description="Comparação reproduzível de embeddings no HateBR."
    )
    subparsers = parser.add_subparsers(dest="command", required=True)
    subparsers.add_parser(
        "listar-modelos",
        help="Mostra modelos, suporte linguístico e uso de instruções.",
    )

    audit = subparsers.add_parser("auditar", help="Valida e resume a base HateBR.")
    audit.add_argument("--dados", type=Path, default=root / "HateBR.csv")

    run = subparsers.add_parser("executar", help="Gera embeddings e compara resultados.")
    run.add_argument("--dados", type=Path, default=root / "HateBR.csv")
    run.add_argument("--cache", type=Path, default=root / "cache")
    run.add_argument("--saida", type=Path, default=root / "outputs")
    run.add_argument(
        "--modelos",
        nargs="+",
        default=list(DEFAULT_MODELS),
        help="Nomes do registro. Use 'listar-modelos' para ver opções.",
    )
    run.add_argument(
        "--condicoes",
        nargs="+",
        default=list(DEFAULT_CONDITIONS),
        choices=TODAS_CONDICOES,
    )
    run.add_argument("--seed", type=int, default=SEED)
    run.add_argument("--batch-size", type=int, default=32)
    run.add_argument("--dispositivo", default="auto", choices=("auto", "cpu", "cuda"))
    run.add_argument("--forcar-embeddings", action="store_true")
    run.add_argument("--groq-model", default="llama-3.1-8b-instant")
    run.add_argument("--bootstrap", type=int, default=500)
    return parser


def main(argv: list[str] | None = None) -> int:
    parser = build_parser()
    args = parser.parse_args(argv)

    if args.command == "listar-modelos":
        for config in MODEL_REGISTRY.values():
            instruction = "sim" if config.supports_instructions else "não"
            print(
                f"{config.name}\n"
                f"  ID: {config.hf_id}\n"
                f"  Idiomas: {config.language_scope}\n"
                f"  Instruções livres: {instruction}\n"
                f"  Papel: {config.role}\n"
                f"  Fonte: {config.source_url}\n"
            )
        return 0

    if args.command == "auditar":
        dataset = load_dataset(args.dados)
        print(json.dumps(dataset.audit, ensure_ascii=False, indent=2))
        print(f"sha256: {dataset.sha256}")
        return 0

    if args.command == "executar":
        model_names = validate_model_names(args.modelos)
        conditions = validate_conditions(args.condicoes)
        results = run_experiment(
            data_path=args.dados,
            cache_dir=args.cache,
            output_dir=args.saida,
            model_names=model_names,
            conditions=conditions,
            seed=args.seed,
            batch_size=args.batch_size,
            device=args.dispositivo,
            force_embeddings=args.forcar_embeddings,
            groq_model=args.groq_model,
            bootstrap_samples=args.bootstrap,
        )
        display_columns = [
            "modelo",
            "condicao",
            "f1_macro",
            "f1_macro_ci95_low",
            "f1_macro_ci95_high",
            "accuracy",
        ]
        print("\nRanking final:")
        print(
            results[
                [column for column in display_columns if column in results.columns]
            ].to_string(index=False)
        )
        return 0

    parser.error("Comando inválido.")
    return 2


if __name__ == "__main__":
    running_in_notebook = (
        not globals().get("__file__") or "ipykernel" in sys.modules
    )
    if running_in_notebook:
        print_colab_instructions()
    else:
        raise SystemExit(main())


In [ ]:
from projeto_pln import executar_colab

resultados = executar_colab(
    modelos=[
        "e5_base",
        "e5_large_instruct",
        "minilm_multilingual",
    ],
    condicoes=[
        "sem_instrucao",
        "manual_en",
        "manual_pt",
    ],
    dispositivo="auto",
)

display(resultados)